# 🛡️ PhishGuard AI — Interactive Pipeline Demonstration
### NLP-Based Phishing & Malicious Email Detection and Risk Analysis System

**Author:** Muhammad Haris  
**Capstone Project:** KPITB AI/ML Training Program  
**Taxonomy:** 3 Classes (`LEGITIMATE`, `PHISHING`, `MALICIOUS`)  

---
This notebook provides a complete interactive walkthrough of the **PhishGuard AI** pipeline:
1. **Data Inspection**: Unified dataset and split distributions
2. **NLP Preprocessing**: Text normalization and token transformation
3. **Cybersecurity Features**: Static indicator extraction and scoring
4. **Model Architecture**: Feature fusion and trained linear baselines
5. **Evaluation**: Real performance metrics and confusion matrix
6. **Inference & Explainability**: Real-time risk analysis on sample emails


In [ ]:
# Setup & Imports
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import pandas as pd
import numpy as np
from src.preprocessing.text_preprocessor import TextPreprocessor
from src.features.security_features import SecurityFeatureExtractor
from src.inference.engine import PhishGuardInference
from src.utils.config import DATA_PROCESSED_DIR, MODELS_DIR

print('Environment initialized successfully!')
print(f'Project root: {project_root}')


## 1. Dataset Inspection & Split Distributions
Review the stratified train, validation, and test splits generated from the verified HuggingFace and Zenodo datasets.

In [ ]:
train_df = pd.read_csv(DATA_PROCESSED_DIR / 'train.csv')
val_df = pd.read_csv(DATA_PROCESSED_DIR / 'val.csv')
test_df = pd.read_csv(DATA_PROCESSED_DIR / 'test.csv')

split_summary = pd.DataFrame({
    'Train (70%)': train_df['label'].value_counts(),
    'Val (10%)': val_df['label'].value_counts(),
    'Test (20%)': test_df['label'].value_counts(),
    'Total': train_df['label'].value_counts() + val_df['label'].value_counts() + test_df['label'].value_counts()
})

print('=== PhishGuard AI Dataset Splits ===')
display(split_summary)
print(f'\nTotal Unified Samples: {len(train_df) + len(val_df) + len(test_df):,}')


## 2. NLP Preprocessing Pipeline
Demonstrating the transformation from raw email content containing HTML, URLs, and noisy formatting into normalized tokens.

In [ ]:
sample_raw_email = """
<html>
<body>
  <p><b>URGENT:</b> Your Microsoft 365 session has expired!</p>
  <p>Please verify your password immediately at <a href="http://192.168.1.1/login">https://login-portal.xyz/verify</a></p>
  <p>Failure to respond within 24 hours will result in permanent suspension.</p>
</body>
</html>
"""

preprocessor = TextPreprocessor(remove_stopwords=False, lemmatize=True)
cleaned_text = preprocessor.preprocess(sample_raw_email)

print('=== RAW EMAIL TEXT ===')
print(sample_raw_email.strip())
print('\n=== PREPROCESSED NLP TEXT ===')
print(cleaned_text)


## 3. Domain-Specific Cybersecurity Feature Extraction
Extracting 17 static security indicators covering urgency, credential harvesting, threat keywords, suspicious TLDs, and executable mentions.

In [ ]:
extractor = SecurityFeatureExtractor()
features = extractor.extract(sample_raw_email)

print(f'Detected Threat Indicators ({len(features.detected_indicators)}):')
for ind in features.detected_indicators:
    print(f'  [!] {ind}')

print('\nExtracted Numerical Feature Vector:')
feature_dict = features.to_dict()
feat_df = pd.DataFrame([feature_dict]).T.rename(columns={0: 'Value'})
display(feat_df)


## 4. Benchmark Model Evaluation Results
Reviewing the performance across all 5 experimental configurations (E1 to E5) and the winning model on the held-out test set.

In [ ]:
results_path = MODELS_DIR / 'experiment_results.json'
if results_path.exists():
    with open(results_path) as f:
        exp_data = json.load(f)
    
    exp_df = pd.DataFrame([
        {
            'Experiment': e['name'],
            'Model': e['model'],
            'Accuracy': f"{e['accuracy']:.4f}",
            'Macro F1': f"{e['macro_f1']:.4f}",
            'Weighted F1': f"{e['weighted_f1']:.4f}",
        }
        for e in exp_data['experiments']
    ])
    print(f"Best Experiment Selected: {exp_data['best_experiment']}")
    display(exp_df)
    
    print('\n=== FINAL TEST SET EVALUATION ===')
    test_res = exp_data['test_results']
    print(f"Test Accuracy:    {test_res['accuracy']:.4f}")
    print(f"Test Macro F1:    {test_res['macro_f1']:.4f}")
    print(f"Test Weighted F1: {test_res['weighted_f1']:.4f}")
else:
    print('Run scripts/train.py first to generate experimental results.')


## 5. Live Inference & Explainable Risk Analysis
Testing the end-to-end inference engine with three representative email scenarios: Legitimate, Phishing, and Malicious.

In [ ]:
try:
    engine = PhishGuardInference()
    
    test_cases = {
        'Legitimate Email': (
            'Hi Team, please find the quarterly financial presentation attached for your review. '
            'Let me know if you have any questions before Thursday meeting. Best regards, Sarah'
        ),
        'Phishing Email': (
            'URGENT NOTIFICATION: Your bank account access has been restricted due to suspicious activity. '
            'Click here immediately to confirm your password and restore access within 24 hours: '
            'http://verify-bank-portal.xyz/login'
        ),
        'Malicious Scareware/Malware': (
            'WARNING: Your system is infected with severe spyware trojans! '
            'Download and run the attached patch update_security.exe immediately to prevent complete drive encryption. '
            'Do not restart your computer.'
        )
    }
    
    for category, email_content in test_cases.items():
        print('=' * 75)
        print(f'TEST CASE: {category}')
        print('=' * 75)
        result = engine.predict(email_content)
        print(f'Predicted Class:  {result.predicted_class} (Confidence: {result.confidence:.1%})')
        print(f'Risk Assessment:  {result.risk_level} (Score: {result.risk_score:.3f})')
        print(f'Class Probabilities:')
        for cls, prob in result.class_probabilities.items():
            print(f'  - {cls}: {prob:.1%}')
        print(f'Detected Threat Triggers: {result.contributing_factors}')
        print(f'Analyst Explanation: {result.explanation}\n')
except Exception as e:
    print(f'Inference engine error: {e}')


## 6. Summary & Defense Readiness
- **Reproducible**: Seed 42, deterministic TF-IDF and Linear SVM.
- **Robust**: Leakage-free split, preprocessed static indicators.
- **Operationally Sound**: <5ms CPU latency, transparent explainability.